# Implementing an Earley Parser

The chart is shared with the pure recognizer. Tree reconstruction enumerates the finite acyclic derivations represented by completed items. Ambiguous grammars produce several trees. A nullable cycle can describe infinitely many trees; revisiting the same nonterminal span is therefore omitted. Recognition still terminates for such grammars.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "EarleyPure.fsx"
open EarleyPure
open GrammarSyntax

`EarleyItem`, prediction, completion, scanning and the grammar loader are defined in the preceding **pure recognizer** notebook. The setup exports its teaching cells as `EarleyPure.fsx`; we reuse the same chart here.

In [ ]:
type ParseTree = TerminalLeaf of string | ProductionNode of string * ParseTree list

// Completed items index the possible nonterminal spans. An active span prevents
// following a cyclic nullable derivation indefinitely while retaining finite trees.
let buildTrees grammar (tokens: Token list) (chart: Set<EarleyItem> array) =
    let input = List.toArray tokens
    let rec variable active name start finish =
        let key = name, start, finish
        if Set.contains key active then []
        else
            chart[finish]
            |> Seq.filter (fun item -> isComplete item && item.Origin = start && item.Rule.Head = name)
            |> Seq.map (fun item -> item.Rule)
            |> Seq.distinct
            |> Seq.collect (fun rule ->
                body (Set.add key active) rule.Body start finish
                |> List.map (fun children -> ProductionNode(name, children)))
            |> Seq.distinct |> Seq.toList
    and body active symbols start finish =
        match symbols with
        | [] -> if start = finish then [[]] else []
        | Terminal kind :: tail ->
            if start < finish && input[start].Kind = kind then
                body active tail (start + 1) finish
                |> List.map (fun children -> TerminalLeaf input[start].Lexeme :: children)
            else []
        | Nonterminal name :: tail ->
            [ for split in start .. finish do
                for child in variable active name start split do
                    for children in body active tail split finish do
                        yield child :: children ]
    variable Set.empty grammar.Start 0 input.Length

let parseTrees grammar tokens =
    let chart = earleyParse grammar tokens
    if Set.contains (finishItem grammar) chart[tokens.Length] then buildTrees grammar tokens chart
    else []

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

let rec toTree = function
    | TerminalLeaf token -> Leaf token
    | ProductionNode(name, children) -> Node(name, List.map toTree children)
let drawTree tree = treeToDot (toTree tree)

In [ ]:
let testParser file word =
    let trees = parseTrees (Parse.file file) (tokenize word)
    printfn "%d parse tree(s) for %s" trees.Length word
    trees |> List.iter (drawTree >> printfn "%s")
    trees

In [ ]:
checkNotebook ((testParser "simple.g" "1 + 2 * 3").Length = 1)

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "simple.g")

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "ambiguous.g")

In [ ]:
checkNotebook ((testParser "ambiguous.g" "1 + 2 * 3").Length = 2)
checkNotebook ((parseTrees (Parse.file "simple.g") (tokenize "1 + * 2")).IsEmpty)